# SepsisWatch on Colab
Run all cells top to bottom. Heavy steps are **cached in Google Drive** (`results/`), so the next session
only loads them (seconds). Set `FORCE = True` to recompute.

**One-time setup:** upload the project folder (without `.venv/`) to Drive as `MyDrive/Sepsis_ML_project`
(with `data/raw/Dataset.csv` inside).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/Sepsis_ML_project'   # change if you used another folder
FORCE = False                                           # True = recompute everything

In [ ]:
import os, sys
os.chdir(PROJECT); sys.path.insert(0, os.path.join(PROJECT, 'src'))
!pip install -q lightgbm shap pyarrow
print(os.listdir('.'))

## 1. Leakage tests (must pass)

In [ ]:
!pip install -q pytest
!cd {PROJECT} && PYTHONPATH=src python -m pytest tests -q

## 2. Features (built once, then loaded from `results/X.parquet`)

In [ ]:
from sepsis.features import cached_features
X, meta = cached_features(force=FORCE)
print(X.shape, meta.Patient_ID.nunique(), 'patients')

## 3. Patient-grouped 5-fold CV (skips folds already saved in Drive)

In [ ]:
from sepsis import train
train.main(force=FORCE)

## 3b. Baselines (SIRS rule, logistic regression)

In [ ]:
from sepsis import baselines
import os
if FORCE or not os.path.exists('results/cv_baseline_logreg_h6.csv'):
    baselines.main()

## 4. Original pipeline replica (optional, ~10 min the first time)

In [ ]:
import os
if FORCE or not os.path.exists('results/replica_original.json'):
    from sepsis import replica; replica.main()
print(open('results/replica_original.json').read())

## 5. Analysis: bootstrap CI, SHAP, figures

In [ ]:
from sepsis import eda, analysis
eda.main(); analysis.main()

In [ ]:
import json, glob
from IPython.display import Image, display
print(json.dumps(json.load(open('results/summary.json'))['bootstrap_ci_h6'], indent=1))
for f in sorted(glob.glob('figures/fig*.png')):
    print(f); display(Image(f))

## 6. Final model (for the API) and monitoring reference

In [ ]:
import os
if FORCE or not os.path.exists('models/lgbm_final.txt'):
    from sepsis import final; final.main()
print(open('models/model_meta.json').read()[:300])